# 02 — Training, optimisation and evaluation

## 1. Purpose

**What:** train, optimise and evaluate the sentiment models (negative / neutral / positive)
on the cleaned Financial PhraseBank data. This notebook is extended step by step: each
modelling iteration gets its own section and one row in `outputs/metrics/iterations.csv`.

**Why:** a fixed, documented protocol makes the iterations comparable and the final result
trustworthy.

> **Integrity rule:** the 20 % test set stays untouched until the final evaluation.
> All model choices (features, hyperparameters, model type) are made with cross-validation
> on the training part only. The test set is evaluated exactly once, for the final model.

In [1]:
# Imports and shared settings
import pandas as pd
from sklearn.dummy import DummyClassifier

from newsentiment import config
from newsentiment.data import load_training_data
from newsentiment.evaluation import create_split, cross_validate_model, load_split, log_iteration

LABELS = list(config.LABELS)  # fixed class order: negative, neutral, positive

## 2. Data and split

**What:** load the cleaned data, create the stratified 80/20 split and apply it.
`create_split()` writes `data/split.json` (row positions plus a fingerprint of the data) on the
first run and simply reuses the file afterwards; `load_split()` refuses to apply it if the data
has changed.

**Why:** a stored split guarantees that every iteration, and the final evaluation, sees exactly
the same training and test rows. Stratification keeps the class shares equal in both parts, which
matters because the classes are imbalanced.

In [2]:
# Cleaned training data (see 01_eda.ipynb for the cleaning steps)
df, report = load_training_data()

# Create the split on the first run, reuse data/split.json afterwards
split = create_split(df)
train_df, test_df = load_split(df)

print(f"Rows: {len(df):,} | split file: {config.SPLIT_FILE.relative_to(config.PROJECT_ROOT).as_posix()}")

Rows: 4,836 | split file: data/split.json


In [3]:
# Size and class counts per part; the shares should be (almost) identical
parts = {"full": df, "train": train_df, "test": test_df}
split_table = pd.DataFrame(
    {name: part["label"].value_counts().reindex(LABELS) for name, part in parts.items()}
).T
split_table["total"] = split_table.sum(axis=1)
for label in LABELS:
    split_table[f"{label}_%"] = (split_table[label] / split_table["total"] * 100).round(1)
split_table

label,negative,neutral,positive,total,negative_%,neutral_%,positive_%
full,604,2871,1361,4836,12.5,59.4,28.1
train,483,2296,1089,3868,12.5,59.4,28.2
test,121,575,272,968,12.5,59.4,28.1


## 3. Evaluation protocol

**What:** every candidate model is scored with **5-fold stratified cross-validation on the
training part** (shuffled, fixed seed, so all models see identical folds). We report mean ± standard
deviation over the folds.

**Main metric: macro-F1** — the unweighted mean of the per-class F1 scores. With about 59 %
neutral sentences, accuracy rewards a model that mostly predicts "neutral"; macro-F1 gives the
small negative class the same weight as the others. Accuracy is reported as a secondary metric.
For the final model, per-class precision/recall and the confusion matrix are added
(`evaluate()` in `newsentiment.evaluation`).

## 4. Iteration "baseline": majority class

**What:** a `DummyClassifier` that always predicts the most frequent class (neutral), ignoring
the text.

**Why:** it is the lower bound every real model must clearly beat. Its accuracy equals the
neutral share, while its macro-F1 is low (F1 = 0 for negative and positive), which shows why
accuracy alone would be misleading.

In [4]:
# Cross-validate the majority-class baseline on the training part only
baseline = DummyClassifier(strategy="most_frequent")
cv_baseline = cross_validate_model(baseline, train_df["text"], train_df["label"])

print(f"Accuracy: {cv_baseline['accuracy_mean']:.3f} ± {cv_baseline['accuracy_std']:.3f}")
print(f"Macro-F1: {cv_baseline['macro_f1_mean']:.3f} ± {cv_baseline['macro_f1_std']:.3f}")

# Per-fold scores
pd.DataFrame(
    {"accuracy": cv_baseline["fold_accuracy"], "macro_f1": cv_baseline["fold_macro_f1"]},
    index=pd.RangeIndex(1, config.CV_FOLDS + 1, name="fold"),
).round(4)

Accuracy: 0.594 ± 0.000
Macro-F1: 0.248 ± 0.000


,accuracy,macro_f1
fold,,
1,0.5943,0.2485
2,0.5930,0.2482
3,0.5930,0.2482
4,0.5938,0.2484
5,0.5938,0.2484


In [5]:
# Log the iteration (re-running replaces the existing "baseline" row)
iterations = log_iteration(
    "baseline",
    "Majority class",
    "DummyClassifier(strategy='most_frequent'); lower bound for all later models",
    cv_baseline,
    baseline.get_params(),
)
iterations

,iteration,model,description,cv_accuracy_mean,cv_accuracy_std,cv_macro_f1_mean,cv_macro_f1_std,params,logged_at
0,baseline,Majority class,DummyClassifier(strategy='most_frequent'); low...,0.5936,0.0005,0.2483,0.0001,"{""constant"": null, ""random_state"": null, ""stra...",2026-11-04T20:28:50
